In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
import os, shutil

/kaggle/input/datasets/dogcdt/synapse/Synapse/test_vol_h5/case0036.npy.h5
/kaggle/input/datasets/dogcdt/synapse/Synapse/test_vol_h5/case0038.npy.h5
/kaggle/input/datasets/dogcdt/synapse/Synapse/test_vol_h5/case0002.npy.h5
/kaggle/input/datasets/dogcdt/synapse/Synapse/test_vol_h5/case0032.npy.h5
/kaggle/input/datasets/dogcdt/synapse/Synapse/test_vol_h5/case0025.npy.h5
/kaggle/input/datasets/dogcdt/synapse/Synapse/test_vol_h5/case0001.npy.h5
/kaggle/input/datasets/dogcdt/synapse/Synapse/test_vol_h5/case0035.npy.h5
/kaggle/input/datasets/dogcdt/synapse/Synapse/test_vol_h5/case0003.npy.h5
/kaggle/input/datasets/dogcdt/synapse/Synapse/test_vol_h5/case0008.npy.h5
/kaggle/input/datasets/dogcdt/synapse/Synapse/test_vol_h5/case0004.npy.h5
/kaggle/input/datasets/dogcdt/synapse/Synapse/test_vol_h5/case0022.npy.h5
/kaggle/input/datasets/dogcdt/synapse/Synapse/test_vol_h5/case0029.npy.h5
/kaggle/input/datasets/dogcdt/synapse/Synapse/train_npz/case0009_slice141.npz
/kaggle/input/datasets/dogcdt/syna

In [2]:
# Setup — run once per session
!pip install -q timm einops ml-collections medpy SimpleITK tensorboardX

!cp -r /kaggle/input/datasets/deepsotaai/adada-transunet-code/DA-TransUNet /kaggle/working/DA-TransUNet
!cp -r /kaggle/input/datasets/deepsotaai/vit-pretrained-weights/model       /kaggle/working/model

# Kaggle strips '+' from filenames — rename back (|| true silences error if already correct)
!mv /kaggle/working/model/vit_checkpoint/imagenet21k/R50ViT-B_16.npz \
    /kaggle/working/model/vit_checkpoint/imagenet21k/R50+ViT-B_16.npz 2>/dev/null || true

# Prevent HuggingFace 'datasets' library from shadowing local datasets/ folder
!touch /kaggle/working/DA-TransUNet/datasets/__init__.py

# Symlink Synapse data (train.py hardcodes ../data/Synapse/ and ignores --root_path)
!mkdir -p /kaggle/working/data/Synapse
!ln -sfn /kaggle/input/datasets/dogcdt/synapse/Synapse/train_npz  /kaggle/working/data/Synapse/train_npz
!ln -sfn /kaggle/input/datasets/dogcdt/synapse/Synapse/test_vol_h5 /kaggle/working/data/Synapse/test_vol_h5

print('Setup complete.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx

In [3]:
%%bash
cd /kaggle/working/DA-TransUNet
python train.py \
  --dataset    Synapse \
  --vit_name   R50-ViT-B_16 \
  --max_epochs 150 \
  --batch_size 24 \
  --base_lr    0.01 \
  --n_skip     3 \
  --img_size   224 \
  --seed       1234

The length of train set is: 2211


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py:186: UserWarning: Initializing zero-element tensors is a no-op
  init.kaiming_uniform_(self.weight, a=math.sqrt(5))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
  0%|                                         | 0/150 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Pl

In [4]:
%%bash
cd /kaggle/working/DA-TransUNet
python test.py \
  --dataset    Synapse \
  --vit_name   R50-ViT-B_16 \
  --num_classes 9 \
  --img_size   224 \
  --is_savenii

Saving results as NIfTI files.
Namespace(volume_path='../data/Synapse/test_vol_h5', dataset='Synapse', num_classes=9, list_dir='./lists/lists_Synapse', max_iterations=20000, max_epochs=150, batch_size=24, img_size=224, is_savenii=True, n_skip=3, vit_name='R50-ViT-B_16', test_save_dir='../predictions', deterministic=1, base_lr=0.01, seed=1234, vit_patches_size=16, Dataset=<class 'datasets.dataset_synapse.Synapse_dataset'>, z_spacing=1, is_pretrain=True, exp='TU_Synapse224')
TU_pretrain_R50-ViT-B_16_skip3_epo150_bs24_224
12 test iterations per epoch
idx 0 case case0008 mean_dice 0.681130 mean_hd95 10.714428
idx 1 case case0022 mean_dice 0.910022 mean_hd95 2.368856
idx 2 case case0038 mean_dice 0.780820 mean_hd95 42.542835
idx 3 case case0036 mean_dice 0.821204 mean_hd95 14.953744
idx 4 case case0032 mean_dice 0.823389 mean_hd95 14.568541
idx 5 case case0002 mean_dice 0.862792 mean_hd95 4.630746
idx 6 case case0029 mean_dice 0.713099 mean_hd95 43.148377
idx 7 case case0003 mean_dice 0.707

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py:186: UserWarning: Initializing zero-element tensors is a no-op
  init.kaiming_uniform_(self.weight, a=math.sqrt(5))
12it [20:44, 103.73s/it]
